In [3]:
import json
import csv

# Map each (model, context_variant) pair to its JSON file path.
FILES = {
    ("GPT4o",       "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json",
    ("GPT4o",       "Method"):  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json",
    ("GPT4o",       "Class"):   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Method"):  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Class"):   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Method"):  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Class"):   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json",
}

OUTPUT_CSV = "instanceNotCompiled.csv"
# ── END CONFIG ────────────────────────────────────────────────────────────────


def analyze(filepath: str) -> dict:
    """
    Load a JSON file whose top-level structure is:
        { "file_counts": { "BBC01": { "files_generated": 1, "files_compiled": 0 }, ... } }
    Returns total bump instances and count where files_compiled == 0.
    """
    with open(filepath, "r") as f:
        data = json.load(f)

    file_counts = data.get("file_counts", data)  # support both wrapped and bare dicts
    total        = sum(1 for v in file_counts.values() if v.get("files_generated", 0) > 0)
    not_compiled = sum(1 for v in file_counts.values() if v.get("files_generated", 0) > 0 and v.get("files_compiled", -1) == 0)
    return {"total": total, "not_compiled": not_compiled}


def main():
    rows = []
    for (model, context), path in FILES.items():
        try:
            result = analyze(path)
            total, not_compiled = result["total"], result["not_compiled"]
            status = "ok"
        except FileNotFoundError:
            total = not_compiled = None
            status = "FILE NOT FOUND"
        except (KeyError, TypeError, json.JSONDecodeError) as e:
            total = not_compiled = None
            status = f"ERROR: {e}"

        rows.append({
            "model":                  model,
            "context_variant":        context,
            "total_instances":        total,
            "instances_not_compiled": not_compiled,
            "status":                 status,
        })
        print(f"[{status}] {model} / {context}: {not_compiled} / {total} not compiled")

    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["model", "context_variant",
                                                "total_instances", "instances_not_compiled", "status"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[ok] GPT4o / Minimal: 47 / 89 not compiled
[ok] GPT4o / Method: 50 / 89 not compiled
[ok] GPT4o / Class: 50 / 89 not compiled
[ok] Qwen3-480b / Minimal: 53 / 89 not compiled
[ok] Qwen3-480b / Method: 52 / 89 not compiled
[ok] Qwen3-480b / Class: 52 / 89 not compiled
[ok] GPTOSS-120b / Minimal: 65 / 89 not compiled
[ok] GPTOSS-120b / Method: 69 / 89 not compiled
[ok] GPTOSS-120b / Class: 65 / 89 not compiled

Saved → instanceNotCompiled.csv


In [1]:
import json
import csv
import statistics

# ── CONFIG ────────────────────────────────────────────────────────────────────
FILES = {
    ("GPT4o",       "Minimal"): "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json",
    ("GPT4o",       "Method"):  "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json",
    ("GPT4o",       "Class"):   "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Minimal"): "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Method"):  "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json",
    ("Qwen3-480b",  "Class"):   "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Minimal"): "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Method"):  "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json",
    ("GPTOSS-120b", "Class"):   "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json",
}

OUTPUT_CSV = "compile_and_test_stat.csv"
# ── END CONFIG ────────────────────────────────────────────────────────────────


def analyze(filepath: str) -> dict:
    with open(filepath, "r") as f:
        data = json.load(f)

    file_counts        = data.get("file_counts", {})
    compilation_results = data.get("compilation_results", {})

    # ── Compile stats (only instances that generated at least 1 file) ──
    total        = sum(1 for v in file_counts.values() if v.get("files_generated", 0) > 0)
    not_compiled = sum(1 for v in file_counts.values()
                       if v.get("files_generated", 0) > 0 and v.get("files_compiled", -1) == 0)

    # ── Test distribution: tests_in_compiled_files per BBC instance ──
    # Only include instances that compiled at least 1 file (tests are meaningful there)
    test_counts = [
        inst.get("test_counts", {}).get("tests_in_compiled_files", 0)
        for inst in compilation_results.values()
        if isinstance(inst, dict)
        and inst.get("file_counts", {}).get("files_compiled", 0) > 0
    ]

    if test_counts:
        t_min    = min(test_counts)
        t_max    = max(test_counts)
        t_median = statistics.median(test_counts)
    else:
        t_min = t_max = t_median = None

    return {
        "total":        total,
        "not_compiled": not_compiled,
        "t_min":        t_min,
        "t_max":        t_max,
        "t_median":     t_median,
    }


def main():
    rows = []
    for (model, context), path in FILES.items():
        try:
            r = analyze(path)
            status = "ok"
        except FileNotFoundError:
            r = {"total": None, "not_compiled": None, "t_min": None, "t_max": None, "t_median": None}
            status = "FILE NOT FOUND"
        except (KeyError, TypeError, json.JSONDecodeError) as e:
            r = {"total": None, "not_compiled": None, "t_min": None, "t_max": None, "t_median": None}
            status = f"ERROR: {e}"

        rows.append({
            "model":                    model,
            "context_variant":          context,
            "total_instances":          r["total"],
            "instances_not_compiled":   r["not_compiled"],
            "tests_in_compiled_min":    r["t_min"],
            "tests_in_compiled_max":    r["t_max"],
            "tests_in_compiled_median": r["t_median"],
            "status":                   status,
        })
        print(
            f"[{status}] {model} / {context}: "
            f"{r['not_compiled']}/{r['total']} not compiled | "
            f"tests (min/med/max): {r['t_min']} / {r['t_median']} / {r['t_max']}"
        )

    fieldnames = [
        "model", "context_variant",
        "total_instances", "instances_not_compiled",
        "tests_in_compiled_min", "tests_in_compiled_max", "tests_in_compiled_median",
        "status",
    ]
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[ok] GPT4o / Minimal: 47/89 not compiled | tests (min/med/max): 1 / 7.0 / 222
[ok] GPT4o / Method: 50/89 not compiled | tests (min/med/max): 1 / 17 / 359
[ok] GPT4o / Class: 50/89 not compiled | tests (min/med/max): 1 / 17 / 492
[ok] Qwen3-480b / Minimal: 53/89 not compiled | tests (min/med/max): 1 / 14.0 / 291
[ok] Qwen3-480b / Method: 52/89 not compiled | tests (min/med/max): 1 / 25 / 336
[ok] Qwen3-480b / Class: 52/89 not compiled | tests (min/med/max): 1 / 27 / 662
[ok] GPTOSS-120b / Minimal: 65/89 not compiled | tests (min/med/max): 1 / 3.5 / 33
[ok] GPTOSS-120b / Method: 69/89 not compiled | tests (min/med/max): 1 / 10.5 / 49
[ok] GPTOSS-120b / Class: 65/89 not compiled | tests (min/med/max): 1 / 15.5 / 57

Saved → compile_and_test_stat.csv


In [4]:
import json
import csv
import statistics

# ── CONFIG ────────────────────────────────────────────────────────────────────
FILES = {
    ("GPT4o",       "Minimal"): {
        "compile": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/execute_results_pre.json",
    },
    ("GPT4o",       "Method"): {
        "compile": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/execute_results_pre.json",
    },
    ("GPT4o",       "Class"): {
        "compile": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/execute_results_pre.json",
    },
    ("Qwen3-480b",  "Minimal"): {
        "compile": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/execute_results_pre.json",
    },
    ("Qwen3-480b",  "Method"): {
        "compile": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/execute_results_pre.json",
    },
    ("Qwen3-480b",  "Class"): {
        "compile": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/execute_results_pre.json",
    },
    ("GPTOSS-120b", "Minimal"): {
        "compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/execute_results_pre.json",
    },
    ("GPTOSS-120b", "Method"): {
        "compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/execute_results_pre.json",
    },
    ("GPTOSS-120b", "Class"): {
        "compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json",
        "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/execute_results_pre.json",
    },
}

OUTPUT_CSV = "compile_and_test_stats.csv"
# ── END CONFIG ────────────────────────────────────────────────────────────────


def dist(values: list):
    """Return (min, median, max) or (None, None, None) if empty."""
    if not values:
        return None, None, None
    return min(values), statistics.median(values), max(values)


def analyze_compile(filepath: str) -> dict:
    with open(filepath, "r") as f:
        data = json.load(f)

    file_counts         = data.get("file_counts", {})
    compilation_results = data.get("compilation_results", {})

    total        = sum(1 for v in file_counts.values() if v.get("files_generated", 0) > 0)
    not_compiled = sum(1 for v in file_counts.values()
                       if v.get("files_generated", 0) > 0 and v.get("files_compiled", -1) == 0)
    inst_compiled = total - not_compiled

    # per-instance compiled file counts (only instances that compiled ≥1 file)
    compiled_counts = [
        inst.get("test_counts", {}).get("tests_in_compiled_files", 0)
        for inst in compilation_results.values()
        if isinstance(inst, dict)
        and inst.get("file_counts", {}).get("files_compiled", 0) > 0
    ]
    c_min, c_med, c_max = dist(compiled_counts)

    return {
        "total": total,
        "inst_compiled": inst_compiled,
        "c_min": c_min, "c_med": c_med, "c_max": c_max,
    }


def analyze_execute(filepath: str) -> dict:
    with open(filepath, "r") as f:
        data = json.load(f)

    execution_results = data.get("execution_results", data)

    # inst_passed: instances with total_passed > 0
    inst_passed = sum(
        1 for inst in execution_results.values()
        if isinstance(inst, dict)
        and inst.get("summary", {}).get("total_passed", 0) > 0
    )

    # per-instance passed file counts (only instances with ≥1 pass)
    passed_counts = [
        inst["summary"]["total_passed"]
        for inst in execution_results.values()
        if isinstance(inst, dict)
        and inst.get("summary", {}).get("total_passed", 0) > 0
    ]
    p_min, p_med, p_max = dist(passed_counts)

    return {
        "inst_passed": inst_passed,
        "p_min": p_min, "p_med": p_med, "p_max": p_max,
    }


def main():
    rows = []
    for (model, context), paths in FILES.items():
        try:
            c = analyze_compile(paths["compile"])
            status = "ok"
        except Exception as e:
            c = {"total": None, "inst_compiled": None, "c_min": None, "c_med": None, "c_max": None}
            status = f"COMPILE ERROR: {e}"

        try:
            e = analyze_execute(paths["execute"])
        except Exception as ex:
            e = {"inst_passed": None, "p_min": None, "p_med": None, "p_max": None}
            if status == "ok":
                status = f"EXECUTE ERROR: {ex}"

        rows.append({
            "model":            model,
            "context_variant":  context,
            "total_instances":  c["total"],
            # Compilation Rate
            "inst_compiled":    c["inst_compiled"],
            "c_min":            c["c_min"],
            "c_med":            c["c_med"],
            "c_max":            c["c_max"],
            # Pass Rate
            "inst_passed":      e["inst_passed"],
            "p_min":            e["p_min"],
            "p_med":            e["p_med"],
            "p_max":            e["p_max"],
            "status":           status,
        })
        print(
            f"[{status}] {model} / {context}: "
            f"compiled {c['inst_compiled']}/{c['total']} | files min/med/max: {c['c_min']}/{c['c_med']}/{c['c_max']} | "
            f"passed {e['inst_passed']}/{c['total']} | files min/med/max: {e['p_min']}/{e['p_med']}/{e['p_max']}"
        )

    fieldnames = [
        "model", "context_variant", "total_instances",
        "inst_compiled", "c_min", "c_med", "c_max",
        "inst_passed",   "p_min", "p_med", "p_max",
        "status",
    ]
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved → {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[ok] GPT4o / Minimal: compiled 42/89 | files min/med/max: 1/7.0/222 | passed 38/89 | files min/med/max: 1/5.0/47
[ok] GPT4o / Method: compiled 39/89 | files min/med/max: 1/17/359 | passed 35/89 | files min/med/max: 1/7/86
[ok] GPT4o / Class: compiled 39/89 | files min/med/max: 1/17/492 | passed 31/89 | files min/med/max: 1/7/171
[ok] Qwen3-480b / Minimal: compiled 36/89 | files min/med/max: 1/14.0/291 | passed 32/89 | files min/med/max: 1/7.0/96
[ok] Qwen3-480b / Method: compiled 37/89 | files min/med/max: 1/25/336 | passed 34/89 | files min/med/max: 1/5.5/92
[ok] Qwen3-480b / Class: compiled 37/89 | files min/med/max: 1/27/662 | passed 34/89 | files min/med/max: 1/5.5/168
[ok] GPTOSS-120b / Minimal: compiled 24/89 | files min/med/max: 1/3.5/33 | passed 9/89 | files min/med/max: 1/3/5
[ok] GPTOSS-120b / Method: compiled 20/89 | files min/med/max: 1/10.5/49 | passed 17/89 | files min/med/max: 1/5/14
[ok] GPTOSS-120b / Class: compiled 24/89 | files min/med/max: 1/15.5/57 | passed 21/89 |